# RK1828 Chinese-CLIP Image-Text Matching Demo


[Chinese-CLIP](https://github.com/OFA-Sys/Chinese-CLIP) (DAMO Academy) matches
an image against several Chinese text candidates and scores them. The text and
vision encoders are converted separately and both run on the RK1828. On the
sample, the correct caption scores ~0.96 vs ~0.04.

| Step | Content |
|---|---|
| 1 | Download models (from huggingface.co) |
| 2 | Extract the prebuilt demo |
| 3 | Run image-text matching |

## 0. Environment check

Make sure the RKNN3 runtime (`librknn3_api.so`) and the RK1828 device are ready.
If this fails, install the driver/runtime first (the `rknn3-rk182x` DEB package)
and check with `sudo rknn-smi info` that the RK1828 shows up.

In [ ]:
from rknn3 import get_lib, find_devices

_ = get_lib()  # loads /usr/lib/librknn3_api.so; failure means the runtime is not installed

devices = find_devices()
if not devices:
    raise RuntimeError("No RK1828 device found! Check: sudo rknn-smi info")
for dev_id, dev_type in devices:
    print(f"Found device: id={dev_id}  interface={dev_type}")

## 1. Download models

The models for this demo are hosted on Hugging Face (account `qhddz`) and are
downloaded directly from **huggingface.co**. Run the next cell to start.

> Files that already exist (and are non-empty) are **skipped automatically** —
> re-running never downloads them twice.
> If your network cannot reach huggingface.co, set up a proxy first, e.g.
> `export https_proxy=http://<your-proxy>:<port>` before starting Jupyter.

In [ ]:
# (HF repo, path in repo, local path)
FILES = [
    ("chinese-clip-rknn3-rk1828", "demo_x86_64.tar.gz", "demos/chinese_clip/demo_x86_64.tar.gz"),
    ("chinese-clip-rknn3-rk1828", "model/chinese_clip_text.embed.bin", "demos/chinese_clip/model/chinese_clip_text.embed.bin"),
    ("chinese-clip-rknn3-rk1828", "model/chinese_clip_text.rknn", "demos/chinese_clip/model/chinese_clip_text.rknn"),
    ("chinese-clip-rknn3-rk1828", "model/chinese_clip_text.tokenizer.gguf", "demos/chinese_clip/model/chinese_clip_text.tokenizer.gguf"),
    ("chinese-clip-rknn3-rk1828", "model/chinese_clip_text.weight", "demos/chinese_clip/model/chinese_clip_text.weight"),
    ("chinese-clip-rknn3-rk1828", "model/chinese_clip_vision.rknn", "demos/chinese_clip/model/chinese_clip_vision.rknn"),
    ("chinese-clip-rknn3-rk1828", "model/chinese_clip_vision.weight", "demos/chinese_clip/model/chinese_clip_vision.weight"),
    ("chinese-clip-rknn3-rk1828", "model/demo.jpg", "demos/chinese_clip/model/demo.jpg"),
]

In [ ]:
from nb_download import ensure_files

ensure_files(FILES, "hf")

## 2. Extract the prebuilt demo

`demo_x86_64.tar.gz` contains the official x86_64 C++ demo and its runtime
libraries (`lib/`). Hugging Face does not host raw executables, so they are
shipped as a tarball; extract once (re-running skips automatically).

> **Why a C++ demo instead of pure Python?**
> Notebooks 01-05 (classification / detection / FastSAM / pose / segmentation)
> are a single model with one forward pass and simple post-processing, so they
> walk through everything in Python (`rknn3.py`). This demo's pipeline is far
> more complex — chained models, tokenizer, KV cache, sliding-window /
> streaming state, hundreds of lines of orchestration — reimplementing it in
> Python would be costly and risk diverging from the official behavior.
> So we run the official `rknn3-model-zoo` C++ demo directly; source code:
> [github.com/airockchip/rknn3-model-zoo](https://github.com/airockchip/rknn3-model-zoo)
> under `examples/`.

In [ ]:
from nb_download import extract_tarball

extract_tarball("demos/chinese_clip/demo_x86_64.tar.gz", "demos/chinese_clip")

In [ ]:
from nb_demo import run_demo, snapshot, new_files, show_images, show_new_images, show_audio

## 3. Run image-text matching

Input: sample image `model/demo.jpg`; the demo has several built-in Chinese
candidate captions and prints a probability for each.

In [ ]:
show_images(["demos/chinese_clip/model/demo.jpg"], max_width=400)

In [ ]:
DEMO = "demos/chinese_clip"
run_demo(["./rknn_qaclipvit_demo",
          "model/chinese_clip_text.rknn", "model/chinese_clip_text.weight",
          "model/chinese_clip_text.embed.bin", "model/chinese_clip_text.tokenizer.gguf",
          "model/chinese_clip_vision.rknn", "model/chinese_clip_vision.weight",
          "model/demo.jpg", "1"], DEMO)

## Summary

- CLIP-style models map images and text into the same vector space; "matching"
  is just similarity — usable for zero-shot classification / text-to-image search.
- The text side of a multimodal model (tokenizer + embed) must be deployed too,
  see `model/chinese_clip_text.*`.